In [ ]:
# ==========================================
# CELL 1: ENVIRONMENT & DEPENDENCY SETUP
# ==========================================
import os
import sys

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Define Project Architecture
PROJECT_ROOT = '/content/drive/MyDrive/StegRL_Project'
DIRS = {
    "data": os.path.join(PROJECT_ROOT, 'data'),             # Put your .mp4 and .zip here
    "frames_in": os.path.join(PROJECT_ROOT, 'frames_in'),   # Extracted raw frames
    "frames_out": os.path.join(PROJECT_ROOT, 'frames_out'), # Stego frames generated by the Agent
    "models": os.path.join(PROJECT_ROOT, 'models'),         # Saved RL weights
    "logs": os.path.join(PROJECT_ROOT, 'logs')              # TensorBoard metrics
}

# 3. Build a clean workspace
print("Initializing project directories...")
for name, path in DIRS.items():
    os.makedirs(path, exist_ok=True)
    print(f" -> Created/Verified: {path}")

# 4. Install advanced differentiable processing and media tools
# - kornia: Allows us to simulate JPEG/Video compression differentiably so the neural network can backpropagate through it.
# - ffmpeg-python: For precise frame extraction and audio preservation.
print("\nInstalling required libraries...")
!pip install -q kornia ffmpeg-python tensorboard stegano

# 5. Verify Hardware Acceleration
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n[SYSTEM CHECK] PyTorch utilizing device: {device.type.upper()}")
if device.type != 'cuda':
    print("WARNING: Go to Runtime > Change runtime type and select T4 GPU to train the RL agent effectively.")
else:
    print("GPU is active. Ready for Deep Reinforcement Learning.")

print("\n[NEXT STEP] Upload 'video.mp4' and 'payload.zip' to:", DIRS['data'])

Mounted at /content/drive
Initializing project directories...
 -> Created/Verified: /content/drive/MyDrive/StegRL_Project/data
 -> Created/Verified: /content/drive/MyDrive/StegRL_Project/frames_in
 -> Created/Verified: /content/drive/MyDrive/StegRL_Project/frames_out
 -> Created/Verified: /content/drive/MyDrive/StegRL_Project/models
 -> Created/Verified: /content/drive/MyDrive/StegRL_Project/logs

Installing required libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 118.6 MB/s eta 0:00:00

[SYSTEM CHECK] PyTorch utilizing device: CUDA
GPU is active. Ready for Deep Reinforcement Learning.

[NEXT STEP] Upload 'video.mp4' and 'payload.zip' to: /content/drive/MyDrive/StegRL_Project/data


In [ ]:
# ==========================================
# THE ULTIMATE U-NET MASTER CELL (v2)
# ==========================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import kornia.augmentation as K
import torch.optim as optim
from torch.utils.data import DataLoader
import math, os, gc

# --- 0. PURGE ---
torch.cuda.empty_cache()
gc.collect()
print("GPU Purged. Initiating Spatial U-Net Pipeline...")

# --- 1. CAPACITY & MAPPING ---
# message_length is from Cell 2 (~521). We use 32x32 (1024 bits) for mapping.
MAP_SIZE = 32
CAPACITY = MAP_SIZE * MAP_SIZE
print(f"Frames: {num_frames}, Bits/frame: {message_length}")
print(f"U-Net Mapping: {CAPACITY} bits capacity (Padded).")

# --- 2. ARCHITECTURES ---
class UNetEncoder(nn.Module):
    def __init__(self):
        super(UNetEncoder, self).__init__()
        self.enc1 = nn.Conv2d(3 + 1, 64, kernel_size=3, padding=1)
        self.enc2 = nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2)
        self.dec1 = nn.Conv2d(128, 64, kernel_size=3, padding=1)
        self.dec2 = nn.Conv2d(64 + 64, 3, kernel_size=3, padding=1)

    def forward(self, image, message):
        # Reshape bit vector to 32x32 map
        msg_map = message.view(-1, 1, MAP_SIZE, MAP_SIZE)
        # Upscale bits to 256x256 using nearest-neighbor (creates structured blocks)
        msg_upscaled = F.interpolate(msg_map, size=(image.shape[2], image.shape[3]), mode='nearest')

        x1 = F.relu(self.enc1(torch.cat([image, msg_upscaled], dim=1)))
        x2 = F.relu(self.enc2(x1))
        y1 = F.relu(self.dec1(F.interpolate(x2, size=(image.shape[2], image.shape[3]))))
        # Skip connection to preserve high-res detail
        residual = torch.tanh(self.dec2(torch.cat([x1, y1], dim=1))) * 0.1
        return image + residual

class UNetDecoder(nn.Module):
    def __init__(self):
        super(UNetDecoder, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1, stride=2)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((MAP_SIZE, MAP_SIZE)) # 32x32
        self.final = nn.Conv2d(256, 1, kernel_size=1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        return self.final(x).view(x.size(0), -1)

# --- 3. RE-INIT ---
encoder = UNetEncoder().to(device)
decoder = UNetDecoder().to(device)
dataloader = DataLoader(VideoFrameDataset(DIRS['frames_in']), batch_size=8, shuffle=True)
opt = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-4)
criterion_bce = nn.BCEWithLogitsLoss()
criterion_mse = nn.MSELoss()

# --- 4. TRAINING ---
for epoch in range(50):
    # Phase Logic
    if epoch < 10:
        mode, visual_w, use_noise = "residual", 0.0, False
    elif epoch < 15:
        mode, visual_w, use_noise = "full_clean", 100.0, False
    else:
        mode, visual_w, use_noise = "full_noisy", 100.0, True

    err_sum = 0.0

    for imgs, _ in dataloader:
        imgs = imgs.to(device)
        # We train on full 1024-bit random messages
        msgs = torch.randint(0, 2, (imgs.size(0), CAPACITY), dtype=torch.float32).to(device)

        stego = encoder(imgs, msgs)

        # Mode Selection
        if mode == "residual": d_in = stego - imgs
        elif mode == "full_clean": d_in = stego
        else: d_in = K.RandomJPEG(jpeg_quality=(70, 90), p=0.5)(stego)

        opt.zero_grad()
        logits = decoder(d_in)

        # Loss: Payload (BCE) + Visual (MSE)
        loss_p = criterion_bce(logits, msgs)
        loss_v = criterion_mse(stego, imgs)

        (loss_p + visual_w * loss_v).backward()
        opt.step()

        with torch.no_grad():
            preds = (torch.sigmoid(logits) > 0.5).float()
            # We only track error on the bits we actually care about (0 to message_length)
            err_sum += torch.mean(torch.abs(preds[:, :message_length] - msgs[:, :message_length])).item()

    avg_err = err_sum / len(dataloader)
    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/50] | Err: {avg_err:.4f} | Mode: {mode}")

torch.save(encoder.state_dict(), os.path.join(DIRS['models'], 'encoder.pth'))
torch.save(decoder.state_dict(), os.path.join(DIRS['models'], 'decoder.pth'))

GPU Purged. Initiating Spatial U-Net Pipeline...
Frames: 242, Bits/frame: 521
U-Net Mapping: 1024 bits capacity (Padded).
Epoch [1/50] | Err: 0.3823 | Mode: residual
Epoch [2/50] | Err: 0.0016 | Mode: residual
Epoch [4/50] | Err: 0.0000 | Mode: residual
Epoch [6/50] | Err: 0.0000 | Mode: residual
Epoch [8/50] | Err: 0.0000 | Mode: residual
Epoch [10/50] | Err: 0.0000 | Mode: residual
Epoch [12/50] | Err: 0.0186 | Mode: full_clean
Epoch [14/50] | Err: 0.0413 | Mode: full_clean
Epoch [16/50] | Err: 0.0173 | Mode: full_noisy
Epoch [18/50] | Err: 0.0101 | Mode: full_noisy
Epoch [20/50] | Err: 0.0104 | Mode: full_noisy
Epoch [22/50] | Err: 0.0104 | Mode: full_noisy
Epoch [24/50] | Err: 0.0080 | Mode: full_noisy
Epoch [26/50] | Err: 0.0101 | Mode: full_noisy
Epoch [28/50] | Err: 0.0085 | Mode: full_noisy
Epoch [30/50] | Err: 0.0080 | Mode: full_noisy
Epoch [32/50] | Err: 0.0076 | Mode: full_noisy
Epoch [34/50] | Err: 0.0084 | Mode: full_noisy
Epoch [36/50] | Err: 0.0063 | Mode: full_noisy
Ep

In [ ]:
# ==========================================
# CELL 2: REED-SOLOMON PAYLOAD PREPARATION (OPTIMIZED)
# ==========================================
from reedsolo import RSCodec
import numpy as np
import os
import torch
import math

# 1. SETUP PATHS
zip_file_path = os.path.join(DIRS['data'], 'payload.zip')

# 2. OPTIMIZED PROTECTION
# Using 32 parity symbols per block.
# This is much more efficient and will fit our capacity.
rs = RSCodec(32)

print(f"Loading ZIP from: {zip_file_path}")

with open(zip_file_path, 'rb') as f:
    raw_zip_data = f.read()

# 3. APPLY PROTECTION
# This will now result in a much smaller protected file
protected_zip_data = rs.encode(raw_zip_data)

# Convert to bits
payload_bits_ecc = np.unpackbits(np.frombuffer(protected_zip_data, dtype=np.uint8))
payload_tensor = torch.FloatTensor(payload_bits_ecc.copy()).to(device)

# 4. CAPACITY CHECK
# Our limit is 1024 bits per frame
message_length = int(math.ceil(payload_tensor.shape[0] / num_frames))

print("-" * 35)
print(f"Original ZIP:    {len(raw_zip_data)} bytes")
print(f"Protected ZIP:   {len(protected_zip_data)} bytes")
print(f"Payload Density: {message_length} bits per frame")
print("-" * 35)

if message_length > 1024:
    print(f"[CRITICAL] Still too large ({message_length}/1024). Reduce parity further.")
else:
    print(f"[SUCCESS] Payload fits perfectly! Density is {message_length}/1024.")

Loading ZIP from: /content/drive/MyDrive/StegRL_Project/data/payload.zip
-----------------------------------
Original ZIP:    15744 bytes
Protected ZIP:   18016 bytes
Payload Density: 596 bits per frame
-----------------------------------
[SUCCESS] Payload fits perfectly! Density is 596/1024.
Proceed to Cell 7.


In [ ]:
# ==========================================
# CELL 3: RS-AWARE U-NET TRAINING (DEFINITIVE)
# ==========================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import kornia.augmentation as K
import torch.optim as optim
from torch.utils.data import DataLoader
import os
import gc

# --- 0. PRE-FLIGHT CHECKS ---
torch.cuda.empty_cache()
gc.collect()

# Ensure variables from Cell 6 exist
if 'message_length' not in locals() or 'num_frames' not in locals():
    raise NameError("CRITICAL: 'message_length' or 'num_frames' not found. Please run Cell 6 first!")

MAP_SIZE = 32
CAPACITY = MAP_SIZE * MAP_SIZE # 1024 bits
print(f"GPU Purged. Initiating Protected Training...")
print(f"Configuration: {message_length} payload bits + parity inside {CAPACITY}-bit spatial grid.")

# --- 1. ARCHITECTURE DEFINITIONS (REDUNANT FOR SAFETY) ---
class UNetEncoder(nn.Module):
    def __init__(self):
        super(UNetEncoder, self).__init__()
        self.enc1 = nn.Conv2d(3 + 1, 64, kernel_size=3, padding=1)
        self.enc2 = nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2)
        self.dec1 = nn.Conv2d(128, 64, kernel_size=3, padding=1)
        self.dec2 = nn.Conv2d(64 + 64, 3, kernel_size=3, padding=1)

    def forward(self, image, message):
        msg_map = message.view(-1, 1, MAP_SIZE, MAP_SIZE)
        msg_upscaled = F.interpolate(msg_map, size=(image.shape[2], image.shape[3]), mode='nearest')
        x1 = F.relu(self.enc1(torch.cat([image, msg_upscaled], dim=1)))
        x2 = F.relu(self.enc2(x1))
        y1 = F.relu(self.dec1(F.interpolate(x2, size=(image.shape[2], image.shape[3]), mode='bilinear')))
        residual = torch.tanh(self.dec2(torch.cat([x1, y1], dim=1))) * 0.1
        return image + residual

class UNetDecoder(nn.Module):
    def __init__(self):
        super(UNetDecoder, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1, stride=2)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((MAP_SIZE, MAP_SIZE))
        self.final = nn.Conv2d(256, 1, kernel_size=1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        return self.final(x).view(x.size(0), -1)

# --- 2. INITIALIZATION ---
encoder = UNetEncoder().to(device)
decoder = UNetDecoder().to(device)
dataloader = DataLoader(VideoFrameDataset(DIRS['frames_in']), batch_size=8, shuffle=True)
opt = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-4)
criterion_bce = nn.BCEWithLogitsLoss()
criterion_mse = nn.MSELoss()

# --- 3. THE 3-PHASE CURRICULUM ---
print("\nStarting 50-Epoch RS-Aware Run. Go grab a coffee.\n")

for epoch in range(50):
    if epoch < 10:
        mode, visual_w, use_noise = "residual", 0.0, False
    elif epoch < 15:
        mode, visual_w, use_noise = "full_clean", 100.0, False
    else:
        mode, visual_w, use_noise = "full_noisy", 100.0, True

    err_sum = 0.0

    for imgs, _ in dataloader:
        imgs = imgs.to(device)
        batch_size = imgs.size(0)

        # Train on full 1024-bit random messages to build a robust "pipe"
        msgs = torch.randint(0, 2, (batch_size, CAPACITY), dtype=torch.float32).to(device)

        stego = encoder(imgs, msgs)

        # Mode Selection Logic
        if mode == "residual":
            d_in = stego - imgs
        elif mode == "full_clean":
            d_in = stego
        else:
            # Random JPEG compression (MP4 simulation)
            d_in = K.RandomJPEG(jpeg_quality=(70, 90), p=0.5)(stego)

        opt.zero_grad()
        logits = decoder(d_in)

        loss_payload = criterion_bce(logits, msgs)
        loss_visual = criterion_mse(stego, imgs)

        (loss_payload + (visual_w * loss_visual)).backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), 1.0)
        opt.step()

        with torch.no_grad():
            preds = (torch.sigmoid(logits) > 0.5).float()
            # Track error specifically on the protected payload slice
            err_sum += torch.mean(torch.abs(preds[:, :message_length] - msgs[:, :message_length])).item()

    # Progress Logging
    if (epoch + 1) % 5 == 0 or epoch == 0:
        avg_err = err_sum / len(dataloader)
        print(f"Epoch [{epoch+1}/50] | Mode: {mode:10} | Avg Bit Error: {avg_err:.4f}")

# --- 4. PERSISTENCE ---
torch.save(encoder.state_dict(), os.path.join(DIRS['models'], 'encoder_rs.pth'))
torch.save(decoder.state_dict(), os.path.join(DIRS['models'], 'decoder_rs.pth'))
print("\n[SUCCESS] Protected Weights Saved. Ready for Cell 8 extraction.")

GPU Purged. Initiating Protected Training...
Configuration: 596 payload bits + parity inside 1024-bit spatial grid.

Starting 50-Epoch RS-Aware Run. Go grab a coffee.

Epoch [1/50] | Mode: residual   | Avg Bit Error: 0.3297
Epoch [5/50] | Mode: residual   | Avg Bit Error: 0.0000
Epoch [10/50] | Mode: residual   | Avg Bit Error: 0.0000
Epoch [15/50] | Mode: full_clean | Avg Bit Error: 0.0064
Epoch [20/50] | Mode: full_noisy | Avg Bit Error: 0.0051
Epoch [25/50] | Mode: full_noisy | Avg Bit Error: 0.0016
Epoch [30/50] | Mode: full_noisy | Avg Bit Error: 0.0011
Epoch [35/50] | Mode: full_noisy | Avg Bit Error: 0.0007
Epoch [40/50] | Mode: full_noisy | Avg Bit Error: 0.0008
Epoch [45/50] | Mode: full_noisy | Avg Bit Error: 0.0009
Epoch [50/50] | Mode: full_noisy | Avg Bit Error: 0.0009

[SUCCESS] Protected Weights Saved. Ready for Cell 8 extraction.


In [ ]:
# ==========================================
# CELL 4: THE BOTTOM-LEFT PATCH STRATEGY
# ==========================================
import cv2
import numpy as np
import glob
import os
from PIL import Image
from torchvision import transforms
import torch
import torch.nn as nn
import torch.nn.functional as F
import ffmpeg

print("Step 1: Loading Golden 256x256 Weights...")

# Standard 256x256 Blueprint
class UNetEncoder(nn.Module):
    def __init__(self):
        super(UNetEncoder, self).__init__()
        self.enc1 = nn.Conv2d(3 + 1, 64, kernel_size=3, padding=1)
        self.enc2 = nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2)
        self.dec1 = nn.Conv2d(128, 64, kernel_size=3, padding=1)
        self.dec2 = nn.Conv2d(64 + 64, 3, kernel_size=3, padding=1)
    def forward(self, image, message):
        msg_map = message.view(-1, 1, 32, 32)
        msg_upscaled = F.interpolate(msg_map, size=(image.shape[2], image.shape[3]), mode='nearest')
        x1 = F.relu(self.enc1(torch.cat([image, msg_upscaled], dim=1)))
        x2 = F.relu(self.enc2(x1))
        y1 = F.relu(self.dec1(F.interpolate(x2, size=(image.shape[2], image.shape[3]), mode='bilinear')))
        residual = torch.tanh(self.dec2(torch.cat([x1, y1], dim=1))) * 0.1
        return image + residual

class UNetDecoder(nn.Module):
    def __init__(self):
        super(UNetDecoder, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1, stride=2)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((32, 32))
        self.final = nn.Conv2d(256, 1, kernel_size=1)
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        return self.final(x).view(x.size(0), -1)

encoder_patch = UNetEncoder().to(device)
decoder_patch = UNetDecoder().to(device)
encoder_patch.load_state_dict(torch.load(os.path.join(DIRS['models'], 'encoder_rs.pth')))
decoder_patch.load_state_dict(torch.load(os.path.join(DIRS['models'], 'decoder_rs.pth')))
encoder_patch.eval()
decoder_patch.eval()

# Patch Coordinates (Bottom-Left Corner)
NATIVE_W, NATIVE_H = 848, 478
PATCH_SIZE = 256
X_START = 0
X_END = PATCH_SIZE
Y_START = NATIVE_H - PATCH_SIZE  # 478 - 256 = 222
Y_END = NATIVE_H                 # 478

transform_to_tensor = transforms.ToTensor()
transform_to_pil = transforms.ToPILImage()

print(f"\nStep 2: Patching frames at X:[{X_START}:{X_END}], Y:[{Y_START}:{Y_END}]...")
with torch.no_grad():
    all_frames = sorted(glob.glob(os.path.join(DIRS['frames_in'], "*.png")))
    for i in range(num_frames):
        # 1. Load full native frame
        full_img = Image.open(all_frames[i]).convert('RGB')
        full_tensor = transform_to_tensor(full_img) # Shape: [3, 478, 848]

        # 2. Extract the bottom-left patch
        patch_tensor = full_tensor[:, Y_START:Y_END, X_START:X_END].unsqueeze(0).to(device)

        # 3. Prepare payload chunk
        start_bit = i * message_length
        end_bit = (i + 1) * message_length
        chunk = payload_tensor[start_bit:end_bit]
        padding = torch.zeros(1024 - len(chunk)).to(device)
        msg_input = torch.cat([chunk, padding]).unsqueeze(0)

        # 4. Encode the patch
        stego_patch = encoder_patch(patch_tensor, msg_input)

        # 5. Hard Replace the patch back into the full frame tensor
        full_tensor[:, Y_START:Y_END, X_START:X_END] = stego_patch.squeeze(0).cpu().clamp(0, 1)

        # 6. Save the native-resolution stego frame
        save_img = transform_to_pil(full_tensor)
        save_img.save(os.path.join(DIRS['frames_out'], f"{i:04d}.png"))

print("\nStep 3: Compiling Native Widescreen MP4...")
output_video_path = os.path.join(DIRS['data'], 'final_patched_widescreen.mp4')
(
    ffmpeg
    .input(os.path.join(DIRS['frames_out'], '%04d.png'), framerate=fps)
    .output(ffmpeg.input(audio_file), output_video_path,
            vcodec='libx264', crf=1, pix_fmt='yuv420p', preset='veryslow', acodec='copy')
    .overwrite_output().run(quiet=True)
)

print("\nStep 4: Extracting Patches from MP4 and Validating...")
cap = cv2.VideoCapture(output_video_path)
recovered_bits = []
f_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret or f_idx >= num_frames: break

    # 1. Read full MP4 frame
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    full_tensor = transform_to_tensor(Image.fromarray(frame_rgb))

    # 2. Crop the bottom-left patch precisely
    patch_t = full_tensor[:, Y_START:Y_END, X_START:X_END].unsqueeze(0).to(device)

    # 3. Decode the patch
    with torch.no_grad():
        logits = decoder_patch(patch_t)
        preds = (torch.sigmoid(logits) > 0.5).float()
        recovered_bits.append(preds[0, :message_length].cpu())
    f_idx += 1
cap.release()

print("\nStep 5: Mathematical Error Correction...")
all_recovered = torch.cat(recovered_bits)[:payload_tensor.shape[0]]
recovered_np = all_recovered.numpy().astype(np.uint8)

raw_errors = np.sum(recovered_np != payload_bits_ecc)
print(f" -> RAW Bit Errors: {raw_errors} (BER: {raw_errors / len(payload_bits_ecc):.4f})")

try:
    recovered_bytes_raw = np.packbits(recovered_np).tobytes()
    fixed_zip_data, _, _ = rs.decode(recovered_bytes_raw)

    final_zip_path = os.path.join(DIRS['data'], 'PERFECT_PATCHED_PAYLOAD.zip')
    with open(final_zip_path, 'wb') as f:
        f.write(fixed_zip_data)

    print(f"\n[SYSTEM SECURED] Patched Widescreen ZIP is 100% healthy: {final_zip_path}")
except Exception as e:
    print(f"\n[CRITICAL ERROR] RS Healing failed: {e}")

Step 1: Loading Golden 256x256 Weights...

Step 2: Patching frames at X:[0:256], Y:[222:478]...

Step 3: Compiling Native Widescreen MP4...

Step 4: Extracting Patches from MP4 and Validating...

Step 5: Mathematical Error Correction...
 -> RAW Bit Errors: 56 (BER: 0.0004)

[SYSTEM SECURED] Patched Widescreen ZIP is 100% healthy: /content/drive/MyDrive/StegRL_Project/data/PERFECT_PATCHED_PAYLOAD.zip


In [ ]:
# ==========================================
# STEGANOGRAPHY EXTRACTION SCRIPT
# ==========================================
# Run this cell to extract the hidden ZIP from the video.

import os
print("Installing dependencies...")
os.system('pip install -q reedsolo')

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from reedsolo import RSCodec

# --- 1. CONFIGURATION (The Lock & Key) ---
VIDEO_PATH = 'final_patched_widescreen.mp4'
WEIGHTS_PATH = 'decoder_rs.pth'

TOTAL_BITS = 144128
MESSAGE_LENGTH = 596
NUM_FRAMES = 242

# --- 2. DECODER BLUEPRINT ---
class UNetDecoder(nn.Module):
    def __init__(self):
        super(UNetDecoder, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1, stride=2)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((32, 32))
        self.final = nn.Conv2d(256, 1, kernel_size=1)
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        return self.final(x).view(x.size(0), -1)

print("\nInitializing Decoder Architecture...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
decoder = UNetDecoder().to(device)
decoder.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
decoder.eval()

transform = transforms.ToTensor()

# --- 3. VIDEO EXTRACTION ---
print("Scanning video for steganographic signature...")
cap = cv2.VideoCapture(VIDEO_PATH)
recovered_bits = []
f_idx = 0

# Patch Coordinates (Bottom-Left 256x256 of the 848x478 video)
NATIVE_H = 478
PATCH_SIZE = 256
Y_START = NATIVE_H - PATCH_SIZE
Y_END = NATIVE_H
X_START = 0
X_END = PATCH_SIZE

while cap.isOpened():
    ret, frame = cap.read()
    if not ret or f_idx >= NUM_FRAMES: break

    # Isolate the exact patch where the data is hidden
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    patch_img = frame_rgb[Y_START:Y_END, X_START:X_END]

    patch_t = transform(patch_img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = decoder(patch_t)
        preds = (torch.sigmoid(logits) > 0.5).float()
        recovered_bits.append(preds[0, :MESSAGE_LENGTH].cpu())
    f_idx += 1
cap.release()

# --- 4. ERROR CORRECTION & RECOVERY ---
print("Applying Reed-Solomon Cryptographic Healing...")
all_recovered = torch.cat(recovered_bits)[:TOTAL_BITS]
recovered_np = all_recovered.numpy().astype(np.uint8)

rs = RSCodec(32) # Standard 32-byte parity armor
recovered_bytes_raw = np.packbits(recovered_np).tobytes()

try:
    fixed_zip_data, _, _ = rs.decode(recovered_bytes_raw)

    with open('EXTRACTED_SECRET.zip', 'wb') as f:
        f.write(fixed_zip_data)

    print("\n[SUCCESS] ZIP file extracted perfectly!")
    print("Check your Colab files folder for 'EXTRACTED_SECRET.zip'")
except Exception as e:
    print(f"\n[FAILURE] The payload was corrupted. Error: {e}")

TOTAL_BITS = 144128
MESSAGE_LENGTH = 596
NUM_FRAMES = 242
